# [ベースライン basic 版] 第1回データ分析コンペティション：NFL Draft Prediction

　本ノートブックでは、「スポーツパフォーマンステストおよび選手情報をもとに、NFLドラフト指名の有無（Drafted）を予測する」タスクのベースラインモデルを構築します。

　この basic 版は、機械学習コンペティションを初めて体験する方向けに、データの読み込みから提出ファイルの作成までの一連の流れを丁寧に体験できるように構成しています。

　本ノートブックを上から順番に実行していけば、提出可能な予測ファイルが作成されます。

## 目次
1. セットアップ
2. データ読み込み
3. データの分析・EDA
4. 前処理
5. ベースラインモデル
6. 仮説と特徴量エンジニアリング
7. 提出ファイル作成
8. 今後の展望

## 1. セットアップ

　まずは Google Drive をマウントします。**マウント** とは、Google Drive のファイルを Colab 上のフォルダのように扱えるようにする設定です。実行するとアクセス許可を求められるので、画面の指示に従って許可してください。

In [1]:
# Google Driveのマウント（Colab上で自分のデータにアクセスするための設定です）
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# モジュールのインポート
import numpy as np  # 数値計算や配列操作を行うためのライブラリ
import pandas as pd  # 表形式のデータを扱うためのライブラリ
import matplotlib.pyplot as plt  # データ可視化のための基本的なグラフ描画ライブラリ
import seaborn as sns  # 高機能な統計グラフを描画するライブラリ
from sklearn.preprocessing import LabelEncoder  # カテゴリ変数を数値に変換するエンコーダ
from sklearn.ensemble import RandomForestClassifier  # ランダムフォレストによる分類器
from sklearn.model_selection import train_test_split  # データ分割を行う関数
from sklearn.metrics import roc_auc_score  # ROC AUCスコアを計算する評価指標

### データ配置

このノートブックではGoogle Drive内において、以下のデータ配置を想定しています。

```
マイドライブ/
└── GCI/
    └── competition_1/
        ├── baseline_basic.ipynb  # このファイル
        └── data/
            ├── train.csv  # 学習用データ
            ├── test.csv  # 予測対象データ
            └── sample_submission.csv  # 提出用サンプルファイル
```

## 2. データ読み込み

　以下のコードでデータを読み込みます。`PATH` の `/content/drive` 部分は Colab で Drive がマウントされる固定の場所、`My Drive` 以下が自分の Drive のフォルダ構成です。

In [3]:
# 読み込むデータが格納されたディレクトリのパス（※必要に応じて変更してください）


train = pd.read_csv("/content/drive/MyDrive/（内部）GCI2026_Summer/03.（内部）コンペ/01.（内部→受講生公開へ移動）コンペ1/data/train.csv")  # 学習用データの読み込み
test = pd.read_csv("/content/drive/MyDrive/（内部）GCI2026_Summer/03.（内部）コンペ/01.（内部→受講生公開へ移動）コンペ1/data/test.csv")    # テスト用データの読み込み

　まずはデータのサイズを確認してみましょう。

In [4]:
print('Train:', train.shape)
print('Test:', test.shape)

Train: (2433, 16)
Test: (1044, 15)


　訓練データとテストデータの行数・列数が表示されます。
　テストデータの列数が訓練データより1つ少ないのは、予測対象の `Drafted` 列がテストデータには含まれていないためです。

　機械学習コンペでは、

- train: 入力特徴量と正解（目的変数）のセット
- test: 入力特徴量のみ。ここに対する予測を提出する

という形になっていることが多いです。

　次に、訓練データの先頭5行を見てみましょう。`.head()` はデフォルトで先頭 5 行を返します（`train.head(10)` で 10 行にもできます）。

In [5]:
train.head()

,Id,Year,Age,School,Height,Weight,Sprint_40yd,Vertical_Jump,Bench_Press_Reps,Broad_Jump,Agility_3cone,Shuttle,Player_Type,Position_Type,Position,Drafted
0,0,2013,22.0,Alabama,1.9050,146.510335,4.94,NaN,30.0,NaN,NaN,NaN,defense,defensive_lineman,DT,1
1,1,2018,21.0,Arizona St.,1.8796,110.676538,4.75,81.28,28.0,289.56,7.03,4.25,defense,line_backer,ILB,1
2,2,2014,23.0,Florida St.,1.9304,142.881597,5.44,NaN,NaN,NaN,NaN,NaN,offense,offensive_lineman,C,1
3,3,2009,22.0,USC,1.8796,102.965468,4.57,NaN,25.0,NaN,NaN,NaN,special_teams,kicking_specialist,K,1
4,4,2018,22.0,Iowa,1.7780,87.996920,4.54,81.28,12.0,NaN,NaN,NaN,offense,backs_receivers,RB,0


　pandas の DataFrame は `.info()` で列名・データ型・欠損の有無などを一度に確認できます。

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2433 entries, 0 to 2432
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Id                2433 non-null   int64  
 1   Year              2433 non-null   int64  
 2   Age               2040 non-null   float64
 3   School            2433 non-null   object 
 4   Height            2433 non-null   float64
 5   Weight            2433 non-null   float64
 6   Sprint_40yd       2316 non-null   float64
 7   Vertical_Jump     1946 non-null   float64
 8   Bench_Press_Reps  1786 non-null   float64
 9   Broad_Jump        1921 non-null   float64
 10  Agility_3cone     1566 non-null   float64
 11  Shuttle           1617 non-null   float64
 12  Player_Type       2433 non-null   object 
 13  Position_Type     2433 non-null   object 
 14  Position          2433 non-null   object 
 15  Drafted           2433 non-null   int64  
dtypes: float64(9), int64(3), object(4)
memory 

　`.info()` の出力では、列ごとに次の情報が見られます。

- **Non-Null Count**：欠損していないデータの件数（全件数より小さい列は欠損あり）
- **Dtype**：データの型。`int64` / `float64` は数値、`object` は文字列（カテゴリ）

## 3. データの分析・EDA

　**EDA（探索的データ分析）** は、モデルを作る前にデータの中身を観察して把握する作業です。

　EDA を通じて、たとえば次のようなことを確認します。

- 欠損値はどこにあるか
- 数値の分布はどうなっているか
- カテゴリごとに件数や傾向に偏りはあるか
- 目的変数とどのように関係していそうか

　EDA を行うことで、後の前処理やモデル設計の方針が見えやすくなります。

　まずは欠損値を確認しましょう。多くの機械学習モデルは欠損値があるとそのままでは学習できないため、必ず確認しておきたいポイントです。

　書き方の補足：`train.isnull()` はマス目ごとに「欠損なら True」の表を返し、`.sum()` は列ごとにその True の数を数えます。`.isnull().sum()` でひとまとめに「列ごとの欠損数」が得られます。

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

　`Age`、`Sprint_40yd`、`Vertical_Jump`、`Bench_Press_Reps`、`Broad_Jump`、`Agility_3cone`、`Shuttle` 列に欠損値があることが分かりました。これらは後の前処理で対処します。

　次に、目的変数 `Drafted` の分布を見てみましょう。`value_counts()` で各値（0 と 1）の件数を数えられます。

In [ ]:
drafted_counts = train['Drafted'].value_counts()

plt.figure(figsize=(8, 6))
plt.bar(drafted_counts.index.astype(str), drafted_counts.values)
plt.title('Distribution of Drafted', fontsize=16)
plt.xlabel('Drafted', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

　ドラフト指名された（1）／されなかった（0）の件数比を確認します。割合も見てみましょう。`value_counts(normalize=True)` のように `normalize=True` をつけると、件数ではなく割合を返します。

In [ ]:
drafted_percentage = train['Drafted'].value_counts(normalize=True) * 100
print(f"Percentage of 0: {drafted_percentage.get(0, 0):.2f}%")
print(f"Percentage of 1: {drafted_percentage.get(1, 0):.2f}%")

　ドラフト指名された選手が約 65 ％、されなかった選手が約 35 ％という、やや不均衡なデータであることが分かりました。

　次に、数値特徴量の分布を確認します。Id と Drafted を除く各列について、ヒストグラムを並べて表示します。

In [ ]:
# 数値列だけを取り出す（Id, Drafted列は除く）
numeric_cols = train.select_dtypes(include=['number']).columns
numeric_cols = numeric_cols.drop(['Id', 'Drafted'])

# プロット
num_cols = len(numeric_cols)
cols = 3
rows = (num_cols + cols - 1) // cols

plt.figure(figsize=(5 * cols, 4 * rows))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(rows, cols, i)
    plt.hist(train[col].dropna(), bins=30, edgecolor='black')
    plt.title(f'Histogram of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

　ヒストグラムを見るときは、「分布が偏っていないか」「外れ値がないか」 **「`Sprint_40yd` だけ他と分布の形が違わないか」** などをざっくり眺めると、後の処理のヒントになります。

　次にカテゴリデータについて、それぞれ何種類の値があるか（水準数）を見てみましょう。

In [ ]:
# カテゴリデータを抽出
categorical_cols = train.select_dtypes(include=['object', 'category']).columns

# 各列の水準数を取得
for col in categorical_cols:
    print(f"{col}: {train[col].nunique()} levels")

　`School` は水準数が非常に多く可視化が難しいので、ここでは他のカテゴリ変数を可視化します。

　ここでは件数を数えてグラフ化する `sns.countplot`（seaborn の関数）を使います。`plt.bar` と違って、自動でカテゴリを集計してくれる点が便利です。`fig, axes = plt.subplots(...)` は複数のグラフを横に並べる定型パターンです。

In [ ]:
# カテゴリデータ（object型）から School を除いて可視化
plot_cols = [c for c in categorical_cols if c != 'School']

fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 5))
if len(plot_cols) == 1:
    axes = [axes]

for i, col in enumerate(plot_cols):
    sns.countplot(x=col, data=train, order=train[col].value_counts().index, ax=axes[i])
    axes[i].set_title(f'Count Plot of {col}', fontsize=14)
    axes[i].set_xlabel(col, fontsize=12)
    axes[i].set_ylabel('Count', fontsize=12)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

　水準ごとに件数の偏りがあることが分かります。

　次に、各カテゴリ水準ごとの `Drafted` 平均（＝ドラフト指名されている割合）を見てみましょう。これにより、どのグループの選手がドラフト指名されやすそうかをざっくり確認できます。

　ここで使う `groupby(col)['Drafted'].mean()` は、「`col` の値ごとにグループ分けして、各グループの `Drafted` の平均を計算する」という処理です。今回は 0/1 の平均なので、結果は「そのグループでドラフト指名された割合」になります。

In [ ]:
fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 5))
if len(plot_cols) == 1:
    axes = [axes]

for i, col in enumerate(plot_cols):
    mean_values = train.groupby(col)['Drafted'].mean().sort_values(ascending=False)
    sns.barplot(x=mean_values.index, y=mean_values.values, ax=axes[i])
    axes[i].set_title(f'Average Drafted by {col}', fontsize=14)
    axes[i].set_xlabel(col, fontsize=12)
    axes[i].set_ylabel('Average Drafted', fontsize=12)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

　`Player_Type` が `special_teams`、`Position_Type` が `kicking_specialist`、`Position` が `K`/`P`/`LS` の選手はドラフト指名割合が他のグループより低いことが見て取れます。

　EDA で得られたこうした観察は、後の前処理や特徴量設計のヒントになります。

## 4. 前処理

　ここでは、モデルに入れやすい形にデータを整えます。

　この basic 版では、シンプルに進めるために次の方針をとります。

- `Id` は識別子なので削除
- 水準数の多い `School` はこの段階では削除
- 数値の欠損値は **train の平均値** で補完
- カテゴリ変数（`Player_Type`, `Position_Type`, `Position`）は **Label Encoding** で数値化

In [ ]:
# 使わない列の削除
train = train.drop(columns=["Id", "School"])
test = test.drop(columns=["Id", "School"])

　次に、欠損値を平均値で補完します。

In [ ]:
# 平均で補完する対象の列
cols_to_fill = ['Age', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                'Broad_Jump', 'Agility_3cone', 'Shuttle']

# train の平均で train/test 両方を補完
for col in cols_to_fill:
    mean_value = train[col].mean()
    train[col] = train[col].fillna(mean_value)
    test[col] = test[col].fillna(mean_value)

　**なぜ train の平均で test も埋めるのか？**　今回のコンペでは test の特徴量を前処理に使うことも許可されているので、`pd.concat([train, test])` 全体の平均値で埋める方法も選択肢の 1 つです。

　ただし、本ノートブックでは「test = 未知のデータ」と捉えて、train の情報のみを使う方針で進めます。実務では学習時に test（未来のデータ）の情報を使えないことが多く、もし誤って使ってしまうと評価結果が実態よりよく見えてしまいます。これを **データリーク** と呼びます。実務感覚に近い習慣をつける意味でも、basic 版では train だけで前処理する形にしています。

　欠損値が消えたことを確認します。

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

　次に、カテゴリ変数を数値に変換します。多くの機械学習モデルは入力として数値データのみを受け付けるため、文字列のままだと学習できません。

　ここでは **Label Encoding** という、各カテゴリに整数 ID を割り当てる手法を使います。たとえば `Player_Type` 列で `offense → 0`, `defense → 1`, `special_teams → 2` のように変換されます。

　**なぜ train と test を結合（`pd.concat`）してから fit するのか？**　test にだけ存在するカテゴリがあった場合に、train だけで fit するとそのカテゴリを変換できずエラーになります。両方を結合してから対応関係を学習させることでこれを防ぎます。

In [ ]:
# カテゴリデータをラベルエンコーディング
label_encoders = {}
for c in ["Player_Type", "Position_Type", "Position"]:
    label_encoders[c] = LabelEncoder()
    label_encoders[c].fit(pd.concat([train[c], test[c]]).astype(str))
    train[c] = label_encoders[c].transform(train[c].astype(str))
    test[c] = label_encoders[c].transform(test[c].astype(str))

　前処理後の train データを確認します。

In [ ]:
train.head()

## 5. ベースラインモデル

　**ベースラインモデル**とは、機械学習タスクに対してまず最初に作るシンプルなモデルのことです。今後の改善の出発点となります。

　今回は、**ランダムフォレスト（Random Forest）** という決定木ベースのモデルを使用します。多数の決定木を組み合わせて予測する手法で、表形式データに対して扱いやすく、過学習しにくいのが特徴です。

　モデルの性能を確かめるために、train データを「学習用」と「検証用」の 2 つに分けます。学習用データだけで学習し、検証用データで予測の良し悪しを評価することで、モデルが未知のデータでもうまく予測できるかを確認できます。

　今回は train データの 30% を検証用に回します。なお、もう少し進んだ評価方法に **クロスバリデーション** （CV）がありますが、これは advanced 版で扱います。

　以下のコードに出てくる重要なキーワードを軽く整理しておきます。

- **`stratify=y`**：分割時に Drafted の 0/1 の比率を train 側と valid 側で揃える設定。クラス不均衡データで重要です。
- **`random_state`**：乱数のタネ。同じ値を指定すれば、何回実行しても同じ分割・同じモデルになります（再現性のため）。
- **`n_estimators=100`**：ランダムフォレストが作る決定木の本数。多いほど安定するが学習が重くなる。
- **`max_depth=5`**：各決定木の深さの上限。深すぎると過学習しやすい。
- **`n_jobs=-1`**：CPU を全部使って並列計算する設定。学習を速くするためのおまじない。
- **`predict_proba(X)[:, 1]`**：「Drafted=1 である確率」を返す。`predict_proba` は `[0 である確率, 1 である確率]` の 2 列を返すので、`[:, 1]` で 2 列目（=1 の確率）だけ取り出しています。AUC は確率値で評価する指標なので、`predict`（0/1 を直接返す）ではなく `predict_proba` を使います。

In [ ]:
# 特徴量と目的変数に分ける
X = train.drop(columns=["Drafted"])
y = train["Drafted"]

# 学習用と検証用に分割（stratify で Drafted の比率を揃える）
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# モデル作成と学習
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# 検証用データで予測
y_valid_pred_proba = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, y_valid_pred_proba)
print("Validation AUC:", round(auc, 4))

　今回の評価指標である **AUC（Area Under the ROC Curve）** は、0.5 に近いとランダムな予測、1.0 に近いほど良いモデルとされる指標です。

　今回はベースラインとして、まず一定のスコアが出ることが確認できました。

## 6. 仮説と特徴量エンジニアリング

　**特徴量エンジニアリング**とは、元のデータからモデルがより有効に学習できるような新しい列（特徴量）を作るプロセスです。モデルの性能を高めるための重要なステップの 1 つです。

　ここでは、「身長と体重から算出される BMI（肥満度指標）は、選手の体格バランスを表す情報として有益かもしれない」という仮説をもとに、`BMI = 体重 / 身長²` を作ってみます。

In [ ]:
# BMI を新しい特徴量として追加
for df in [train, test]:
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)

　新しい特徴量を加えた状態で、もう一度同じ手順でモデルを学習・評価してみましょう。なお、ここで `model` 変数を再代入するので、以降の予測には BMI を含むモデルが使われます。

　以下のコードはセクション 5 のモデル学習コードとほぼ同じです。違いは、いま `train` と `test` に BMI 列が追加されたので、その新しい特徴量も含めてモデルが学習する点だけです。

In [ ]:
# 特徴量と目的変数に分ける（BMI を含む）
X = train.drop(columns=["Drafted"])
y = train["Drafted"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_valid_pred_proba = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, y_valid_pred_proba)
print("Validation AUC after feature engineering:", round(auc, 4))

　特徴量を追加することで、AUC が変化したことが確認できると思います。

　ただし、特徴量を追加すれば必ずスコアが上がるわけではありません。仮説を立てて、追加してみて、検証する。この **「仮説 → 検証」のサイクル** が機械学習コンペでは重要です。

　ぜひ自分なりの仮説を立てて、新しい特徴量を作ってみてください。

## 7. 提出ファイル作成

　最後に、test データに対して予測を行い、提出用のファイルを作成します。

　コンペでは指定された形式で予測ファイルを提出する必要があります。今回は `sample_submission.csv` を読み込んで、`Drafted` 列を予測値で上書きする形で作ります。

　なお、`to_csv(..., index=False)` の `index=False` は、行番号（pandas のインデックス）を CSV に書き出さない設定です。これがないと提出フォーマットがズレるので必須です。

In [ ]:
# test データへの予測（学習済みモデルを使用）
test_pred_proba = model.predict_proba(test)[:, 1]

# 提出ファイル作成
submission = pd.read_csv(PATH + 'sample_submission.csv')
submission["Drafted"] = test_pred_proba
submission.to_csv(PATH + 'submission_basic.csv', index=False)

　提出ファイルの先頭を確認しましょう。

In [ ]:
submission.head()

　`submission_basic.csv` ができました。これをコンペに提出してみましょう。

## 8. 今後の展望

　本ノートブックでは、データの読み込みから提出ファイル作成までの一連の流れを体験しました。

　さらにスコアを上げたい場合は、次のような工夫が考えられます。

- 別のモデルを試す（例：勾配ブースティング、ニューラルネット）
- カテゴリ変数のエンコーディング手法を変える（One-Hot Encoding、Target Encoding 等）
- 欠損値の補完手法を工夫する（中央値補完、グループ別平均など）
- `School` 列をうまく特徴量として活用する
- ハイパーパラメータを調整する
- さらに新しい特徴量を作る
- **`baseline_advanced.ipynb`** に進んで、改善サイクルの考え方を学ぶ

　まずはこの basic 版の内容をしっかり理解したうえで、advanced 版に進むと改善の流れが分かりやすくなります。